# Week 2 · Day 1 — Window Functions
**Goal:** Rank, compare, and compute running totals without collapsing rows — the most powerful SQL pattern analysts use daily.

**Dataset:** Canadian logistics — shipments, drivers, clients, routes, warehouses

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Week 1 foundation you're bringing in:** JOINs, GROUP BY grain, subqueries, SUM(quantity * unit_price) precision, correlated subqueries

---
## Business First Prompt

> *You've just joined a Canadian logistics company as a data analyst. The operations manager asks: "I want to know which drivers are our top performers by freight revenue, how each shipment ranks within its own route, and whether our freight charges are trending up or down over the year. Can you build that?"*

Write 3–5 sentences below before touching any data:
- Why can't GROUP BY alone answer this?
- What does a window function let you see that an aggregate hides?


**Your answer:**

> GROUP BY alone can’t answer this because it aggregates data into one row per group, which removes the detailed information needed for ranking and trend analysis. Window functions allow you to retain each shipment while also calculating metrics like driver performance rankings, shipment rankings within each route, and changes in freight charges over time (e.g., using functions like RANK() or LAG()). This makes it possible to analyze both overall performance and row-level patterns simultaneously.

---
## Setup — run this first

In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\Users\gianc\Documents\Logistics Operations Project\logistics_w2.db')
print('Connected to logistics_w2.db')

Connected to logistics_w2.db


In [13]:
for table in ['clients','warehouses','vehicles','drivers','routes','shipments','shipment_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- clients ---
Columns: ['client_id', 'company', 'city', 'province', 'segment']


,client_id,company,city,province,segment
0,1,Maple Retail Co,Toronto,ON,Enterprise
1,2,Pacific Goods Ltd,Vancouver,BC,Enterprise
2,3,Prairie Supply Inc,Calgary,AB,SMB



--- warehouses ---
Columns: ['warehouse_id', 'name', 'city', 'province', 'capacity_sqft']


,warehouse_id,name,city,province,capacity_sqft
0,1,Toronto Central,Toronto,ON,85000
1,2,Vancouver Hub,Vancouver,BC,72000
2,3,Calgary Depot,Calgary,AB,55000



--- vehicles ---
Columns: ['vehicle_id', 'plate', 'type', 'max_kg']


...

,route_id,origin_wh_id,dest_city,dest_province,distance_km
0,1,1,Montreal,QC,541.0
1,2,1,Ottawa,ON,450.0
2,3,1,Halifax,NS,1800.0



--- shipments ---
Columns: ['shipment_id', 'client_id', 'route_id', 'driver_id', 'vehicle_id', 'ship_date', 'delivery_date', 'status', 'freight_charge']


,shipment_id,client_id,route_id,driver_id,vehicle_id,ship_date,delivery_date,status,freight_charge
0,1001,4,4,3,2,2024-11-23,2024-11-24,Delivered,3375.74
1,1002,2,10,7,1,2024-12-12,None,Delayed,4030.98
2,1003,10,1,9,4,2024-01-16,2024-01-20,Delivered,1162.07



--- shipment_items ---
Columns: ['item_id', 'shipment_id', 'description', 'category', 'weight_kg', 'declared_value']


,item_id,shipment_id,description,category,weight_kg,declared_value
0,1,1001,Electronics,Standard,787.6,321.17
1,2,1002,Office Supplies,Standard,614.3,3832.60
2,3,1002,Clothing,Standard,97.7,2512.06


---
## Table Preview — run this so you know what you're working with

In [14]:
for table in ['clients','warehouses','vehicles','drivers','routes','shipments','shipment_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- clients ---
Columns: ['client_id', 'company', 'city', 'province', 'segment']


,client_id,company,city,province,segment
0,1,Maple Retail Co,Toronto,ON,Enterprise
1,2,Pacific Goods Ltd,Vancouver,BC,Enterprise
2,3,Prairie Supply Inc,Calgary,AB,SMB



--- warehouses ---
Columns: ['warehouse_id', 'name', 'city', 'province', 'capacity_sqft']


,warehouse_id,name,city,province,capacity_sqft
0,1,Toronto Central,Toronto,ON,85000
1,2,Vancouver Hub,Vancouver,BC,72000
2,3,Calgary Depot,Calgary,AB,55000



--- vehicles ---
Columns: ['vehicle_id', 'plate', 'type', 'max_kg']


...

,route_id,origin_wh_id,dest_city,dest_province,distance_km
0,1,1,Montreal,QC,541.0
1,2,1,Ottawa,ON,450.0
2,3,1,Halifax,NS,1800.0



--- shipments ---
Columns: ['shipment_id', 'client_id', 'route_id', 'driver_id', 'vehicle_id', 'ship_date', 'delivery_date', 'status', 'freight_charge']


,shipment_id,client_id,route_id,driver_id,vehicle_id,ship_date,delivery_date,status,freight_charge
0,1001,4,4,3,2,2024-11-23,2024-11-24,Delivered,3375.74
1,1002,2,10,7,1,2024-12-12,None,Delayed,4030.98
2,1003,10,1,9,4,2024-01-16,2024-01-20,Delivered,1162.07



--- shipment_items ---
Columns: ['item_id', 'shipment_id', 'description', 'category', 'weight_kg', 'declared_value']


,item_id,shipment_id,description,category,weight_kg,declared_value
0,1,1001,Electronics,Standard,787.6,321.17
1,2,1002,Office Supplies,Standard,614.3,3832.60
2,3,1002,Clothing,Standard,97.7,2512.06


---
## Concept 1 — What Is a Window Function?

A window function computes a value **across a set of rows related to the current row** — without collapsing them into a single output row like GROUP BY does.

**GROUP BY vs Window Function:**

| | GROUP BY | Window Function |
|---|---|---|
| Output rows | One per group | Same as input — no rows lost |
| Individual row detail | Gone | Still visible |
| Use case | Totals and summaries | Rank, running total, compare to group |

**Syntax:**
```sql
function_name() OVER (
    PARTITION BY column   -- defines the group (optional)
    ORDER BY column       -- defines the order within the group
)
```

- `PARTITION BY` — splits rows into groups (like GROUP BY, but rows are kept)
- `ORDER BY` inside OVER — determines the order within each partition
- If you omit `PARTITION BY`, the window is the entire result set

**Key window functions:**

| Function | What it does |
|---|---|
| `ROW_NUMBER()` | Unique sequential number per row within partition |
| `RANK()` | Rank with gaps on ties |
| `DENSE_RANK()` | Rank without gaps on ties |
| `SUM() OVER` | Running or partitioned total |
| `AVG() OVER` | Running or partitioned average |
| `LAG()` | Value from the previous row |
| `LEAD()` | Value from the next row |

In [15]:
# EXAMPLE — rank every delivered shipment by freight charge, highest first
# Notice: all rows are kept, each gets a rank number
q = """
SELECT shipment_id,
       driver_id,
       freight_charge,
       ship_date,
       RANK() OVER (ORDER BY freight_charge DESC) AS charge_rank
FROM shipments
WHERE status = 'Delivered'
ORDER BY charge_rank
LIMIT 15
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,driver_id,freight_charge,ship_date,charge_rank
0,1107,6,4485.35,2024-09-28,1
1,1183,8,4476.63,2024-12-30,2
2,1175,3,4445.74,2024-01-02,3
3,1141,7,4416.37,2024-08-21,4
4,1021,2,4377.71,2024-08-20,5
5,1143,8,4349.29,2024-01-19,6
6,1188,2,4311.32,2024-10-11,7
7,1065,6,4277.60,2024-07-05,8
8,1153,7,4261.78,2024-03-21,9
9,1038,7,4180.35,2024-01-02,10


---
## Drill 1 — Rank all delivered shipments by freight charge

Show every delivered shipment with its rank by freight_charge descending.  
Columns: shipment_id, route_id, freight_charge, charge_rank.  

Hint: use `RANK() OVER (ORDER BY freight_charge DESC)`

In [16]:
q1 = """ SELECT shipment_id,
                freight_charge,
                RANK() OVER (ORDER BY freight_charge DESC ) AS freight_amount_rank

 FROM shipments
 WHERE status = 'Delivered'


"""
df = pd.read_sql_query(q1, conn)
display(df)

,shipment_id,freight_charge,freight_amount_rank
0,1107,4485.35,1
1,1183,4476.63,2
2,1175,4445.74,3
3,1141,4416.37,4
4,1021,4377.71,5
...,...,...,...
104,1054,306.72,105
105,1165,258.86,106
106,1163,210.55,107
107,1046,190.52,108


---
## Drill 2 — RANK vs DENSE_RANK: see the difference

Show shipment_id, freight_charge, RANK() and DENSE_RANK() side by side — delivered shipments only.  
Order by freight_charge descending. Limit 20.  

Look at the output: where do the two columns differ and why?

In [17]:
q2 = """ SELECT shipment_id,
                freight_charge,
                RANK() OVER (ORDER BY freight_charge DESC) AS freight_amount_rank,
                DENSE_RANK() OVER (ORDER BY freight_charge DESC ) AS freight_amount_dense_rank

    FROM shipments 
    WHERE status = 'Delivered'

"""
df = pd.read_sql_query(q2, conn)
display(df)

,shipment_id,freight_charge,freight_amount_rank,freight_amount_dense_rank
0,1107,4485.35,1,1
1,1183,4476.63,2,2
2,1175,4445.74,3,3
3,1141,4416.37,4,4
4,1021,4377.71,5,5
...,...,...,...,...
104,1054,306.72,105,105
105,1165,258.86,106,106
106,1163,210.55,107,107
107,1046,190.52,108,108


**Where do RANK and DENSE_RANK differ in your output, and why?**

> *(write here)*

---
## Concept 2 — PARTITION BY: Rank Within Groups

`PARTITION BY` resets the window function for each group — like running the same rank independently inside each partition.

```sql
-- Rank shipments within each route separately
RANK() OVER (PARTITION BY route_id ORDER BY freight_charge DESC)
```

Without PARTITION BY — rank 1 is the single highest across all rows.  
With PARTITION BY route_id — rank 1 is the highest within each route.

```sql
SELECT shipment_id,
       route_id,
       freight_charge,
       RANK() OVER (PARTITION BY route_id ORDER BY freight_charge DESC) AS rank_in_route
FROM shipments
WHERE status = 'Delivered'
```

**Think of PARTITION BY as GROUP BY that doesn't collapse rows.**

In [18]:
# EXAMPLE — rank shipments within each route by freight charge
q = """
SELECT shipment_id,
       route_id,
       freight_charge,
       RANK() OVER (PARTITION BY route_id ORDER BY freight_charge DESC) AS rank_in_route
FROM shipments
WHERE status = 'Delivered'
ORDER BY route_id, rank_in_route
LIMIT 20
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,route_id,freight_charge,rank_in_route
0,1097,1,2350.93,1
1,1051,1,2348.53,2
2,1085,1,1337.27,3
3,1003,1,1162.07,4
4,1179,1,1113.74,5
5,1037,1,1079.80,6
6,1062,1,926.77,7
7,1111,1,700.64,8
8,1143,2,4349.29,1
9,1024,2,3969.34,2


---
## Drill 3 — Rank drivers within each client by freight charge

For each client, rank their shipments by freight_charge descending — delivered only.  
Columns: client_id, shipment_id, driver_id, freight_charge, rank_within_client.  

Hint: PARTITION BY client_id

In [19]:
q3 = """ SELECT s.client_id,
                c.company,
                s.shipment_id,
                s.driver_id,
                s.freight_charge,
                RANK() OVER ( PARTITION BY s.client_id ORDER BY s.freight_charge DESC) AS freight_charge_rank

FROM shipments s JOIN clients c ON s.client_id = c.client_id
WHERE status = 'Delivered'
 
"""
df = pd.read_sql_query(q3, conn)
display(df)

,client_id,company,shipment_id,driver_id,freight_charge,freight_charge_rank
0,1,Maple Retail Co,1064,9,3233.22,1
1,1,Maple Retail Co,1029,2,3229.11,2
2,1,Maple Retail Co,1087,7,2724.73,3
3,1,Maple Retail Co,1068,5,2533.53,4
4,1,Maple Retail Co,1152,5,2149.13,5
...,...,...,...,...,...,...
104,10,Lakeside Goods,1003,9,1162.07,10
105,10,Lakeside Goods,1126,1,1145.82,11
106,10,Lakeside Goods,1179,9,1113.74,12
107,10,Lakeside Goods,1182,10,698.93,13


---
## Drill 4 — Top 1 shipment per route by freight charge

Using PARTITION BY route_id, find only the rank-1 shipment per route — the highest freight charge on each route.  
Columns: route_id, shipment_id, freight_charge.  

Hint: wrap the window function in a subquery, then filter WHERE rank_in_route = 1 in the outer query.

In [20]:
q4 = """ SELECT route_id,
                freight_charge               
FROM ( SELECT shipment_id,
                route_id,
                freight_charge,
                RANK() OVER (PARTITION BY route_id ORDER BY freight_charge DESC) AS freight_rank_shipments
                FROM shipments) AS rank
WHERE freight_rank_shipments = 1
"""
df = pd.read_sql_query(q4, conn)
display(df)

,route_id,freight_charge
0,1,4431.12
1,2,4496.88
2,3,4149.33
3,4,3962.20
4,5,4476.63
5,6,4485.35
6,7,3976.43
7,8,4343.51
8,9,4377.71
9,10,4416.37


---
## Concept 3 — SUM() OVER: Running Totals and Partitioned Totals

`SUM() OVER` without ORDER BY gives a **partitioned total** — the same total repeated on every row in the partition.  
`SUM() OVER` with ORDER BY gives a **running total** — cumulative sum up to the current row.

```sql
-- Partitioned total: total freight for each driver on every row
SUM(freight_charge) OVER (PARTITION BY driver_id)

-- Running total: cumulative freight over time
SUM(freight_charge) OVER (ORDER BY ship_date)

-- Running total per driver over time
SUM(freight_charge) OVER (PARTITION BY driver_id ORDER BY ship_date)
```

**Why partitioned totals are powerful:** you can show each row's value AND the group total on the same row — enabling % of total calculations without a subquery.

In [21]:
# EXAMPLE — each shipment alongside the driver's total freight and % contribution
q = """
SELECT shipment_id,
       driver_id,
       freight_charge,
       SUM(freight_charge) OVER (PARTITION BY driver_id) AS driver_total,
       ROUND(freight_charge * 100.0 / SUM(freight_charge) OVER (PARTITION BY driver_id), 1) AS pct_of_driver_total
FROM shipments
WHERE status = 'Delivered'
ORDER BY driver_id, freight_charge DESC
LIMIT 20
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,driver_id,freight_charge,driver_total,pct_of_driver_total
0,1075,1,3647.88,6682.19,54.6
1,1074,1,1888.49,6682.19,28.3
2,1126,1,1145.82,6682.19,17.1
3,1021,2,4377.71,27143.77,16.1
4,1188,2,4311.32,27143.77,15.9
5,1181,2,3248.95,27143.77,12.0
6,1029,2,3229.11,27143.77,11.9
7,1117,2,2851.47,27143.77,10.5
8,1051,2,2348.53,27143.77,8.7
9,1070,2,2208.36,27143.77,8.1


---
## Drill 5 — Each shipment's freight as % of its route total

Show each delivered shipment alongside the total freight for its route and what percentage of that total it represents.  
Columns: shipment_id, route_id, freight_charge, route_total, pct_of_route.  
Round percentage to 1 decimal place.

In [22]:
q5 = """ SELECT shipment_id,
                route_id,
                freight_charge,
                SUM(freight_charge) OVER (PARTITION BY route_id) AS route_total,
                ROUND(freight_charge*100/SUM(freight_charge) OVER (PARTITION BY route_id), 1)  AS pctg_of_total
    FROM shipments
    WHERE status = 'Delivered'
"""
df = pd.read_sql_query(q5, conn)
display(df)

,shipment_id,route_id,freight_charge,route_total,pctg_of_total
0,1003,1,1162.07,11019.75,10.5
1,1037,1,1079.80,11019.75,9.8
2,1051,1,2348.53,11019.75,21.3
3,1062,1,926.77,11019.75,8.4
4,1085,1,1337.27,11019.75,12.1
...,...,...,...,...,...
104,1079,12,1344.50,12468.81,10.8
105,1098,12,2914.80,12468.81,23.4
106,1117,12,2851.47,12468.81,22.9
107,1125,12,2751.46,12468.81,22.1


---
## Drill 6 — Running total of freight revenue over time

Show each delivered shipment with a running cumulative total of freight_charge ordered by ship_date.  
Columns: ship_date, shipment_id, freight_charge, running_total.  
Order by ship_date ascending.

In [23]:
q6 = """ SELECT ship_date,
                shipment_id,
                freight_charge,
                SUM(freight_charge) OVER (ORDER BY ship_date, shipment_id) AS running_freight_charge

    FROM shipments
    WHERE status = 'Delivered'
    ORDER BY ship_date ASC
"""
df = pd.read_sql_query(q6, conn)
display(df)

,ship_date,shipment_id,freight_charge,running_freight_charge
0,2024-01-02,1038,4180.35,4180.35
1,2024-01-02,1175,4445.74,8626.09
2,2024-01-15,1062,926.77,9552.86
3,2024-01-15,1102,941.00,10493.86
4,2024-01-15,1149,3613.22,14107.08
...,...,...,...,...
104,2024-12-19,1109,1091.27,231624.15
105,2024-12-22,1074,1888.49,233512.64
106,2024-12-23,1112,596.26,234108.90
107,2024-12-26,1009,3026.50,237135.40


---
## Drill 7 — Running total per driver over time

Show each delivered shipment with a running cumulative freight total per driver, ordered by ship_date.  
Columns: driver_id, ship_date, shipment_id, freight_charge, driver_running_total.  

Hint: PARTITION BY driver_id ORDER BY ship_date inside the OVER clause.

In [24]:
q7 = """ SELECT driver_id,
                ship_date,
                shipment_id,
                freight_charge,
                SUM(freight_charge) OVER (PARTITION BY driver_id ORDER BY ship_date) AS running_total
        FROM shipments
        WHERE status = 'Delivered'
"""
df = pd.read_sql_query(q7, conn)
display(df)

,driver_id,ship_date,shipment_id,freight_charge,running_total
0,1,2024-05-08,1126,1145.82,1145.82
1,1,2024-08-05,1075,3647.88,4793.70
2,1,2024-12-22,1074,1888.49,6682.19
3,2,2024-01-15,1062,926.77,926.77
4,2,2024-03-12,1167,1473.99,2400.76
...,...,...,...,...,...
104,10,2024-09-16,1139,1380.35,5145.77
105,10,2024-10-02,1168,2123.35,7269.12
106,10,2024-11-07,1182,698.93,7968.05
107,10,2024-12-11,1165,258.86,8226.91


---
## Concept 4 — LAG() and LEAD(): Compare to Adjacent Rows

`LAG(col, n)` returns the value from `n` rows **before** the current row.  
`LEAD(col, n)` returns the value from `n` rows **after** the current row.  
Default offset is 1 if not specified.

```sql
-- Compare each shipment's freight to the previous one (by date)
SELECT shipment_id,
       ship_date,
       freight_charge,
       LAG(freight_charge) OVER (ORDER BY ship_date) AS prev_charge,
       freight_charge - LAG(freight_charge) OVER (ORDER BY ship_date) AS change
FROM shipments
```

**Use cases:**
- Month-over-month change in revenue
- Days between consecutive shipments for a driver
- Detect if a shipment was more or less expensive than the previous one

**Important:** LAG/LEAD return NULL for the first/last row where there is no previous/next row.

In [25]:
# EXAMPLE — each shipment vs the previous shipment's freight charge on the same route
q = """
SELECT shipment_id,
       route_id,
       ship_date,
       freight_charge,
       LAG(freight_charge) OVER (PARTITION BY route_id ORDER BY ship_date) AS prev_charge_same_route,
       ROUND(freight_charge - LAG(freight_charge) OVER (PARTITION BY route_id ORDER BY ship_date), 2) AS change
FROM shipments
WHERE status = 'Delivered'

LIMIT 20
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,route_id,ship_date,freight_charge,prev_charge_same_route,change
0,1062,1,2024-01-15,926.77,NaN,NaN
1,1003,1,2024-01-16,1162.07,926.77,235.30
2,1179,1,2024-02-25,1113.74,1162.07,-48.33
3,1085,1,2024-04-22,1337.27,1113.74,223.53
4,1111,1,2024-06-03,700.64,1337.27,-636.63
5,1051,1,2024-06-24,2348.53,700.64,1647.89
6,1037,1,2024-07-27,1079.80,2348.53,-1268.73
7,1097,1,2024-12-04,2350.93,1079.80,1271.13
8,1143,2,2024-01-19,4349.29,NaN,NaN
9,1167,2,2024-03-12,1473.99,4349.29,-2875.30


---
## Drill 8 — Each driver's shipment vs their previous shipment freight

For each driver, show each delivered shipment alongside their previous shipment's freight charge and the difference.  
Columns: driver_id, ship_date, shipment_id, freight_charge, prev_freight, change.  
Order by driver_id, ship_date.  

Hint: PARTITION BY driver_id ORDER BY ship_date inside LAG().

In [26]:
q8 = """ SELECT driver_id,
                ship_date,
                shipment_id,
                freight_charge,
                LAG(freight_charge) OVER (Partition BY driver_id ORDER BY ship_date) AS previous_order_freight,
                ROUND(freight_charge - LAG(freight_charge) OVER (PARTITION BY driver_id ORDER BY ship_date),2) AS difference 

    FROM shipments
    WHERE status = 'Delivered'
    
"""
df = pd.read_sql_query(q8, conn)
display(df)

,driver_id,ship_date,shipment_id,freight_charge,previous_order_freight,difference
0,1,2024-05-08,1126,1145.82,NaN,NaN
1,1,2024-08-05,1075,3647.88,1145.82,2502.06
2,1,2024-12-22,1074,1888.49,3647.88,-1759.39
3,2,2024-01-15,1062,926.77,NaN,NaN
4,2,2024-03-12,1167,1473.99,926.77,547.22
...,...,...,...,...,...,...
104,10,2024-09-16,1139,1380.35,665.26,715.09
105,10,2024-10-02,1168,2123.35,1380.35,743.00
106,10,2024-11-07,1182,698.93,2123.35,-1424.42
107,10,2024-12-11,1165,258.86,698.93,-440.07


---
## Drill 9 — Flag shipments where freight dropped vs previous on same route

Using LAG, find delivered shipments where the freight charge is lower than the previous shipment on the same route.  
Columns: route_id, shipment_id, ship_date, freight_charge, prev_charge, drop_amount.  
Order by drop_amount descending (biggest drop first).  

Hint: wrap in a subquery, then filter WHERE freight_charge < prev_charge in the outer query.

In [27]:
q9 = """ SELECT route_id,
                shipment_id,
                ship_date,
                freight_charge,
                prev_charge,
                freight_charge - prev_charge AS change

FROM ( SELECT route_id,
                shipment_id,
                ship_date,
                freight_charge,
                LAG(freight_charge) OVER (PARTITION BY route_id ORDER BY ship_date) AS prev_charge
                FROM shipments
                WHERE status = 'Delivered') sub
WHERE sub.prev_charge > freight_charge
ORDER BY route_id


"""
df = pd.read_sql_query(q9, conn)
display(df)

,route_id,shipment_id,ship_date,freight_charge,prev_charge,change
0,1,1179,2024-02-25,1113.74,1162.07,-48.33
1,1,1111,2024-06-03,700.64,1337.27,-636.63
2,1,1037,2024-07-27,1079.80,2348.53,-1268.73
3,2,1167,2024-03-12,1473.99,4349.29,-2875.30
4,2,1163,2024-07-06,210.55,3969.34,-3758.79
5,2,1100,2024-08-26,1317.52,2048.31,-730.79
6,2,1047,2024-10-02,1303.62,2724.73,-1421.11
7,2,1009,2024-12-26,3026.50,3248.95,-222.45
8,3,1114,2024-02-03,472.26,3800.65,-3328.39
9,3,1186,2024-06-09,3043.22,4034.67,-991.45


---
## Concept 5 — Window Functions with JOINs

Window functions work alongside JOINs — you join first to get the columns you need, then apply the window function in SELECT.

```sql
SELECT d.name,
       s.shipment_id,
       s.freight_charge,
       RANK() OVER (PARTITION BY s.driver_id ORDER BY s.freight_charge DESC) AS rank_for_driver
FROM shipments s
JOIN drivers d ON s.driver_id = d.driver_id
WHERE s.status = 'Delivered'
```

**Rule:** The JOIN happens in FROM first. The window function then sees the full joined result and applies across it. Window functions cannot be in WHERE — use a subquery to filter on window results.

In [28]:
# EXAMPLE — driver name + rank by total freight earned (requires join + window)
q = """
SELECT d.name,
       s.driver_id,
       SUM(s.freight_charge) AS total_freight,
       RANK() OVER (ORDER BY SUM(s.freight_charge) DESC) AS driver_rank
FROM shipments s
JOIN drivers d ON s.driver_id = d.driver_id
WHERE s.status = 'Delivered'
GROUP BY s.driver_id, d.name
ORDER BY driver_rank
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,driver_id,total_freight,driver_rank
0,Sara Singh,3,37379.58,1
1,Aisha Diallo,9,34477.68,2
2,Linda Park,7,33087.99,3
3,Carlos Rivera,6,30453.34,4
4,James Leblanc,8,27825.29,5
5,Mike Okafor,2,27143.77,6
6,Tom Beaumont,4,22524.42,7
7,Priya Mehta,5,12719.59,8
8,Ryan Kowalski,10,9318.18,9
9,Jean Tremblay,1,6682.19,10


---
## Drill 10 — Rank drivers by total delivered freight, include driver name and province

Join shipments to drivers. Show driver name, province, total freight, and their overall rank.  
Columns: name, province, total_freight, driver_rank. Active drivers only.  

Hint: GROUP BY first to get totals, then apply RANK() OVER on the grouped result.

In [29]:
q10 = """ SELECT d.name,
                d.province,
                SUM(s.freight_charge) AS total_freight,
                RANK() OVER (ORDER BY SUM(s.freight_charge) DESC) AS driver_rank
        FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
        WHERE s.status = 'Delivered'
        GROUP BY d.name, d.province
"""
df = pd.read_sql_query(q10, conn)
display(df)

,name,province,total_freight,driver_rank
0,Sara Singh,BC,37379.58,1
1,Aisha Diallo,QC,34477.68,2
2,Linda Park,BC,33087.99,3
3,Carlos Rivera,MB,30453.34,4
4,James Leblanc,NS,27825.29,5
5,Mike Okafor,ON,27143.77,6
6,Tom Beaumont,AB,22524.42,7
7,Priya Mehta,ON,12719.59,8
8,Ryan Kowalski,ON,9318.18,9
9,Jean Tremblay,QC,6682.19,10


---
## Drill 11 — Each client's shipments ranked within their segment

Join shipments to clients. For each shipment, show its rank by freight_charge within the client's segment.  
Columns: company, segment, shipment_id, freight_charge, rank_in_segment.  
Delivered shipments only. Order by segment, rank_in_segment.

Hint: PARTITION BY c.segment ORDER BY freight_charge DESC.

In [30]:
q11 = """ SELECT c.company,
                c.segment,
                s.shipment_id,
                s.freight_charge,
                RANK() OVER (PARTITION BY c.segment ORDER BY freight_charge DESC) AS shipment_rank
    FROM shipments s JOIN clients c ON s.client_id = c.client_id
    WHERE status = 'Delivered' 
  
"""
df = pd.read_sql_query(q11, conn)
display(df)

,company,segment,shipment_id,freight_charge,shipment_rank
0,Pacific Goods Ltd,Enterprise,1080,4034.67,1
1,Pacific Goods Ltd,Enterprise,1024,3969.34,2
2,Pacific Goods Ltd,Enterprise,1035,3954.01,3
3,Rockies Imports,Enterprise,1084,3862.50,4
4,Pacific Goods Ltd,Enterprise,1184,3599.97,5
...,...,...,...,...,...
104,Lakeside Goods,SMB,1182,698.93,55
105,Atlantic Foods,SMB,1172,548.78,56
106,Northern Freight,SMB,1090,376.57,57
107,Northern Freight,SMB,1054,306.72,58


---
## Drill 12 — Top driver per province by total freight

Find the #1 ranked driver in each province by total delivered freight.  
Columns: province, name, total_freight.  

Hint: inner query computes total freight per driver and assigns RANK() OVER (PARTITION BY province). Outer query filters rank = 1.

In [31]:
q12 = """ SELECT province,
                driver_id,
                name,
                 total_freight
FROM (SELECT s.driver_id,
            d.province,
            d.name,
            s.freight_charge,
            SUM(freight_charge) AS  total_freight,
            RANK() OVER (PARTITION BY d.province ORDER BY SUM(s.freight_charge)DESC ) AS rank
        FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
        WHERE s.status= 'Delivered'
        GROUP BY d.province, s.driver_id, d.name) sub
        WHERE sub.rank = 1
   
ORDER BY total_freight DESC
"""
df = pd.read_sql_query(q12, conn)
display(df)

,province,driver_id,name,total_freight
0,BC,3,Sara Singh,37379.58
1,QC,9,Aisha Diallo,34477.68
2,MB,6,Carlos Rivera,30453.34
3,NS,8,James Leblanc,27825.29
4,ON,2,Mike Okafor,27143.77
5,AB,4,Tom Beaumont,22524.42


---
## Block 2 · 11:00am — Applied Challenge

> *The operations manager wants a driver performance dashboard for Q4 2024 (October–December). She needs: who are the top 3 drivers by freight volume, how does each driver's freight per shipment compare to the company average, and is any driver showing a declining trend (last shipment lower than their own average)?*

**Query 1:** Top 3 drivers by total delivered freight in Q4 2024 — include driver name and total shipments  
**Query 2:** Each driver's average freight per shipment vs the company-wide average — flag above or below  
**Query 3:** Drivers whose most recent shipment freight is below their own average (declining signal)

Write your framing sentence first.

**My approach — which window functions and joins are needed for each query:**

> *(write here)*

In [32]:
# Query 1 — Top 3 drivers by Q4 2024 freight
q13 = """ SELECT name, total_freight, driver_rank
FROM (
  SELECT d.name,
         SUM(s.freight_charge) AS total_freight,
         RANK() OVER (ORDER BY SUM(s.freight_charge) DESC) AS driver_rank
  FROM shipments s
  JOIN drivers d ON s.driver_id = d.driver_id
  WHERE s.status = 'Delivered'
  AND s.ship_date >= '2024-10-01'
  AND s.ship_date <= '2024-12-31'
  GROUP BY d.name
) sub
ORDER BY driver_rank
LIMIT 3
"""
df = pd.read_sql_query(q13, conn)
display(df)

,name,total_freight,driver_rank
0,Sara Singh,15807.78,1
1,Aisha Diallo,11850.38,2
2,Mike Okafor,10216.26,3


In [33]:
# Query 2 — Each driver vs company average, flagged
q14 = """ SELECT name,
       total_freight,
       AVG(total_freight) OVER () AS company_avg,
       CASE WHEN total_freight > AVG(total_freight) OVER () 
            THEN 'Above' ELSE 'Below' END AS flag
FROM (
  SELECT d.name,
         SUM(s.freight_charge) AS total_freight
  FROM shipments s
  JOIN drivers d ON s.driver_id = d.driver_id
  WHERE s.status = 'Delivered'  AND s.ship_date >= '2024-10-01'
  AND s.ship_date <= '2024-12-31'
  GROUP BY d.name
) sub
ORDER BY total_freight DESC
"""
df = pd.read_sql_query(q14, conn)
display(df)

,name,total_freight,company_avg,flag
0,Sara Singh,15807.78,7160.125,Above
1,Aisha Diallo,11850.38,7160.125,Above
2,Mike Okafor,10216.26,7160.125,Above
3,James Leblanc,7012.97,7160.125,Below
4,Tom Beaumont,6984.84,7160.125,Below
5,Linda Park,6456.77,7160.125,Below
6,Ryan Kowalski,4172.41,7160.125,Below
7,Carlos Rivera,3771.93,7160.125,Below
8,Priya Mehta,3439.42,7160.125,Below
9,Jean Tremblay,1888.49,7160.125,Below


In [34]:
# Query 3 — Drivers with declining signal (latest < own average)
q15 = """ SELECT    name, 
                    driver_id, 
                    latest_freight, 
                    driver_avg,
                    ROUND(latest_freight - driver_avg, 2) AS difference
                    FROM (  SELECT d.name,
                            s.driver_id,
                            s.freight_charge AS latest_freight,
                            AVG(s.freight_charge) OVER (PARTITION BY s.driver_id) AS driver_avg,
                            ROW_NUMBER() OVER (PARTITION BY s.driver_id ORDER BY s.ship_date DESC) AS rn
                            FROM shipments s
                            JOIN drivers d ON s.driver_id = d.driver_id
                            WHERE s.status = 'Delivered'
                            ) sub
WHERE rn = 1 AND latest_freight < driver_avg
ORDER BY difference
"""
df = pd.read_sql_query(q15, conn)
display(df)

,name,driver_id,latest_freight,driver_avg,difference
0,Aisha Diallo,9,306.72,2298.512000,-1991.79
1,Priya Mehta,5,596.26,1413.287778,-817.03
2,Linda Park,7,2350.93,2757.332500,-406.40
3,Jean Tremblay,1,1888.49,2227.396667,-338.91
4,Ryan Kowalski,10,1091.27,1164.772500,-73.50


**What does the data say? What would you tell the operations manager?**

> *(write here)*

---
## Capstone — Driver Performance Scorecard

Build a single query that produces a full driver scorecard for all delivered shipments. One row per driver showing:

| Column | Definition |
|---|---|
| name | Driver name |
| province | Driver province |
| total_shipments | COUNT of delivered shipments |
| total_freight | SUM of freight_charge |
| avg_freight | Average freight per shipment (rounded to 2dp) |
| overall_rank | RANK() by total_freight descending across all drivers |
| province_rank | RANK() by total_freight within their province |
| vs_company_avg | `'Above'` or `'Below'` company-wide avg freight per shipment |

Order by overall_rank.

Hint: you need GROUP BY for totals, RANK() OVER for both rank columns, AVG() OVER without PARTITION for the company average comparison, and CASE WHEN for the flag.

In [35]:
q_capstone = """ SELECT name,
                        province,
                        total_delivered_shipments,
                        total_freight,
                        avg_freight,
                        RANK() OVER (ORDER BY total_freight DESC) AS freight_rank,
                        RANK() OVER (PARTITION BY province ORDER BY total_freight DESC) AS province_rank,
                        CASE WHEN total_freight > (SELECT AVG(freight_charge) AS  company_avg_freight
                                                            FROM shipments
                                                            WHERE status = 'Delivered') THEN 'Above' ELSE 'Below' END AS vs_company_avg

FROM drivers d
JOIN (
  SELECT driver_id,
         COUNT(shipment_id) AS total_delivered_shipments,
         SUM(freight_charge) AS total_freight,
         AVG(freight_charge) AS avg_freight
  FROM shipments
  WHERE status = 'Delivered'
  GROUP BY driver_id
) ds ON d.driver_id = ds.driver_id
ORDER BY freight_rank;

"""
df = pd.read_sql_query(q_capstone, conn)
display(df)

,name,province,total_delivered_shipments,total_freight,avg_freight,freight_rank,province_rank,vs_company_avg
0,Sara Singh,BC,15,37379.58,2491.972000,1,1,Above
1,Aisha Diallo,QC,15,34477.68,2298.512000,2,1,Above
2,Linda Park,BC,12,33087.99,2757.332500,3,2,Above
3,Carlos Rivera,MB,12,30453.34,2537.778333,4,1,Above
4,James Leblanc,NS,13,27825.29,2140.406923,5,1,Above
5,Mike Okafor,ON,13,27143.77,2087.982308,6,1,Above
6,Tom Beaumont,AB,9,22524.42,2502.713333,7,1,Above
7,Priya Mehta,ON,9,12719.59,1413.287778,8,2,Above
8,Ryan Kowalski,ON,8,9318.18,1164.772500,9,3,Above
9,Jean Tremblay,QC,3,6682.19,2227.396667,10,2,Above


---
## Day 1 Checkpoint — answer before Day 2

Plain English — no SQL needed.

**1. What is the key difference between GROUP BY and a window function? When would you choose each?**

> GROUP BY aggregates data by collapsing multiple rows into a single row per group, meaning you lose the original row-level detail. It’s used when you want summarized results, such as total sales per region. A window function, on the other hand, performs calculations across a set of rows related to the current row but retains all original rows. Each row still appears in the output, with an additional computed value

**2. What does PARTITION BY do inside a window function? How is it different from ORDER BY inside OVER?**

> PARTITION BY divides the result set into separate groups and the window function is applied independently within each of those groups—similar to how GROUP BY creates groups, but without collapsing rows. ORDER BY inside the OVER clause defines the order of rows within each partition, which is important for calculations like rankings or running totals.

**3. You want to find the top driver per province. Why can't you just use GROUP BY and ORDER BY — what do you need instead?**

> You can’t rely on just GROUP BY and ORDER BY because GROUP BY aggregates data but doesn’t allow you to rank or filter the top result within each group (province). To find the top driver per province, you need a window function like RANK() or ROW_NUMBER() with PARTITION BY province and ORDER BY the performance metric (total freight charge DESC). This lets you rank drivers within each province, and then filter for the top-ranked driver.

**4. What does LAG() return for the very first row in a partition, and why?**

> LAG() returns NULL for the first row in a partition because there is no preceding row to retrieve a value from. Since LAG() accesses a previous row, the first row has no prior row, so the result is NULL unless a default value is provided.